### Step 1 — Environment

Run **once per new kernel** (SageMaker notebook instance, local venv, etc.).

- Prefer installing from **`requirements-mlops.txt`** at the repo root so Git + SageMaker jobs share the same dependency list.
- Do **not** `pip install boto3` here (SageMaker: can break the pinned `awscli` / `botocore` stack).


In [ ]:
import subprocess
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Walk upward from cwd until we see repo markers."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "requirements-mlops.txt").exists():
            return p
        if (p / "configs" / "train.yaml").exists():
            return p
    return cwd


_REPO = _find_repo_root()
_REQ = _REPO / "requirements-mlops.txt"

if _REQ.is_file():
    print(f"Installing from {_REQ} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_REQ)])
else:
    print("requirements-mlops.txt not found; using inline pins (same as reference notebook).")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch>=2.2.0",
            "pandas>=2.0.0",
            "numpy>=1.26.0",
            "scikit-learn>=1.4.0",
            "mlflow>=2.10.0",
            "pyarrow>=15.0.0",
            "s3fs>=2024.1.0",
            "polars>=0.20.0",
            "PyYAML>=6.0.0",
        ]
    )

print("Environment setup done.")


## Two-Tower end-to-end orchestration (thin notebook)

This notebook should stay **thin**: load config + call library functions.

All real logic lives in the `two_tower` package under `src/two_tower/`.


In [ ]:
# Dev mode: import `two_tower` from this repo without `pip install -e .`
import sys
from pathlib import Path


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "requirements-mlops.txt").exists():
            return p
        if (p / "configs" / "train.yaml").exists():
            return p
    raise FileNotFoundError(
        "Could not find repo root (looked for requirements-mlops.txt or configs/train.yaml). "
        "Open or `cd` to the Two_Tower repository root and restart the kernel."
    )


repo_root = find_repo_root()
sys.path.insert(0, str(repo_root / "src"))

print("Repo root:", repo_root)
print("Python path[0]:", sys.path[0])


### Step 2 — Load pipeline config (YAML → dataclasses)

Edit paths and feature lists in `configs/train.yaml`, then run the next cell.


In [ ]:
from two_tower.config_loader import load_pipeline_config

TRAIN_CONFIG = (repo_root / "configs" / "train.yaml").resolve()
pipeline_cfg = load_pipeline_config(TRAIN_CONFIG)
print("train path:", pipeline_cfg.paths.train)
print("label_col:", pipeline_cfg.features.label_col)


### Step 3 — Load train / validation Parquet

Uses `pd.read_parquet` on the paths in config (S3 or local). Optional single-client metadata broadcast is controlled by the `data:` section in `train.yaml`.


In [ ]:
from two_tower.data.load import load_train_validation_frames

train_df, val_df = load_train_validation_frames(pipeline_cfg)


### Step 3b — Feature columns, vocabs, hash embedding tables

Intersects configured columns with Parquet, splits cat/num/multi, fits vocabs on **train** only, builds hash init weights (reference notebook). Re-run after changing `features:` or `train.min_count` in YAML.


In [ ]:
from two_tower.features.prepare import prepare_training_features

feature_artifacts = prepare_training_features(train_df, val_df, pipeline_cfg)


### Step 4 — Train + MLflow + artifacts

Runs the full loop (BCE logits, val AUC, best-epoch checkpoint), logs to MLflow, and writes **user tower** `.pt`, **vocab** `.pkl`, and **client embeddings** Parquet under `paths.artifacts_base` (S3 or local). Requires `MLflow` and a valid `mlflow_tracking_uri` / local tracking if you use the server.


In [ ]:
from two_tower.training import train_and_log

train_and_log(
    cfg=pipeline_cfg,
    train_df=train_df,
    val_df=val_df,
    feature_artifacts=feature_artifacts,
)


### Step 5 — Batch inference

Set `paths.infer`, `paths.artifacts_base` (same as training), and `infer.ranking_output` in `configs/infer.yaml`. Artifacts must exist: `artifacts/user_tower/user_tower_state.pt`, `artifacts/vocab_artifact/vocab_artifact.pkl`, `artifacts/client_embeddings/client_embeddings.parquet`.


In [ ]:
from two_tower.config_loader import load_infer_job_config
from two_tower.inference.run import run_inference_job

INFER_CONFIG = (repo_root / "configs" / "infer.yaml").resolve()
infer_cfg = load_infer_job_config(INFER_CONFIG)
run_inference_job(infer_cfg)
